# 03 - Verification Evaluation

Ba setting đánh giá:

1. `claim_only`: chỉ đưa claim vào LLM.
2. `gold_evidence`: đưa claim + gold evidence từ dataset.
3. `retrieved_evidence`: đưa claim + top-5 retrieved evidence từ retrieval/reranker.


In [ ]:
# !pip -q uninstall -y transformers accelerate bitsandbytes tokenizers huggingface-hub safetensors

!pip -q install --no-cache-dir \
  "numpy>=1.26,<2.3" \
  "pandas>=2.1,<2.3" \
  "scikit-learn>=1.4,<1.7" \
  "tqdm>=4.66,<5.0" \
  "sentencepiece>=0.2,<0.3" \
  "safetensors==0.5.3" \
  "huggingface-hub==0.30.2" \
  "tokenizers==0.21.1" \
  "transformers==4.51.3" \
  "accelerate==1.6.0" \
  "bitsandbytes==0.45.5"

In [7]:
import sys
import gc
import re
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import importlib.metadata as md
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

for package_name in ["transformers", "accelerate", "bitsandbytes", "scikit-learn", "pandas", "numpy"]:
    try:
        print(f"{package_name}: {md.version(package_name)}")
    except Exception:
        print(f"{package_name}: not installed")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA: 12.8
CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4
transformers: 4.51.3
accelerate: 1.6.0
bitsandbytes: 0.45.5
scikit-learn: 1.6.1
pandas: 2.2.3
numpy: 2.0.2


In [8]:
# Chỉ đổi biến này khi chuyển từ dev sang test.
RUN_TAG = "dev"  # "dev" hoặc "test"

DATASET_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/dataset/dev_dataset.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/dataset/test_dataset.jsonl",
}

RETRIEVED_EVIDENCE_PATHS = {
    "dev": "/kaggle/input/datasets/lanthanhvi/retrival-output-dev/hybrid_reranker_alpha05_top5_all_labels_dev.jsonl",
    "test": "/kaggle/input/datasets/lanthanhvi/retrival-output/hybrid_alpha05_top5_all_labels_test.jsonl",
}

# Dùng thư mục output mới để tránh lẫn với notebook cũ.
OUTPUT_DIR = Path("/kaggle/working/datasets/lanthanhvi/verification_outputs_clean")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {"tag": "qwen25_3b_8bit", "name": "Qwen/Qwen2.5-3B-Instruct", "quantization": "8bit"},
    {"tag": "qwen25_7b_8bit", "name": "Qwen/Qwen2.5-7B-Instruct", "quantization": "8bit"},
]

SETTINGS = ["claim_only", "gold_evidence", "retrieved_evidence"]

LABELS = ["SUPPORTED", "REFUTED", "NOT_ENOUGH_EVIDENCE"]
EXTENDED_LABELS = LABELS + ["PARSE_ERROR"]

MAX_SAMPLES = None       # Đặt số nhỏ, ví dụ 10, nếu muốn test nhanh.
MAX_NEW_TOKENS = 64
MAX_EVIDENCE_ITEMS = 5
FORCE_RERUN = False      # True nếu muốn chạy lại dù output đã tồn tại.

## 1. Hàm đọc/ghi JSONL và validate schema

Các file của bạn đều là JSONL: mỗi dòng là một JSON object. Vì vậy notebook chỉ giữ hàm đọc JSONL, không còn xử lý nhiều cấu trúc JSON khác nhau.

In [9]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() != ".jsonl":
        raise ValueError(f"Expected .jsonl file, got: {path}")

    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON at {path}, line {line_no}") from exc
            if not isinstance(obj, dict):
                raise ValueError(f"Each JSONL line must be an object. Error at {path}, line {line_no}")
            rows.append(obj)
    return rows


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def require_fields(row, required_fields, row_name):
    missing = [field for field in required_fields if field not in row]
    if missing:
        raise ValueError(f"Missing fields in {row_name}: {missing}")


def validate_dataset(rows):
    required = ["claim_id", "chunk_id", "pages", "raw_text", "claim", "label", "claim_type", "evidence_ids", "split"]
    seen = set()

    for idx, row in enumerate(rows):
        row_name = f"dataset row {idx}"
        require_fields(row, required, row_name)

        claim_id = str(row["claim_id"])
        if claim_id in seen:
            raise ValueError(f"Duplicate claim_id in dataset: {claim_id}")
        seen.add(claim_id)

        if row["label"] not in LABELS:
            raise ValueError(f"Invalid label at {row_name}: {row['label']}")
        if not isinstance(row["evidence_ids"], list) or len(row["evidence_ids"]) == 0:
            raise ValueError(f"evidence_ids must be a non-empty list at {row_name}")
        if not isinstance(row["raw_text"], str) or not row["raw_text"].strip():
            raise ValueError(f"raw_text must be a non-empty string at {row_name}")
        if not isinstance(row["claim"], str) or not row["claim"].strip():
            raise ValueError(f"claim must be a non-empty string at {row_name}")

    return True


def validate_retrieval_rows(rows):
    required_top = ["claim_id", "retrieved_evidence"]
    required_ev = ["rank", "chunk_id", "pages", "raw_text"]
    seen = set()

    for idx, row in enumerate(rows):
        row_name = f"retrieval row {idx}"
        require_fields(row, required_top, row_name)

        claim_id = str(row["claim_id"])
        if claim_id in seen:
            raise ValueError(f"Duplicate claim_id in retrieval file: {claim_id}")
        seen.add(claim_id)

        evidences = row["retrieved_evidence"]
        if not isinstance(evidences, list) or len(evidences) == 0:
            raise ValueError(f"retrieved_evidence must be a non-empty list at {row_name}")

        for ev_idx, ev in enumerate(evidences):
            ev_name = f"retrieval row {idx}, evidence {ev_idx}"
            if not isinstance(ev, dict):
                raise ValueError(f"Evidence must be an object at {ev_name}")
            require_fields(ev, required_ev, ev_name)
            if not isinstance(ev["raw_text"], str) or not ev["raw_text"].strip():
                raise ValueError(f"raw_text must be a non-empty string at {ev_name}")

    return True


def build_retrieved_map(rows):
    # Schema cố định: mỗi row có claim_id và retrieved_evidence.
    return {str(row["claim_id"]): row["retrieved_evidence"] for row in rows}


def validate_dataset_retrieval_match(dataset_rows, retrieved_map):
    dataset_ids = {str(row["claim_id"]) for row in dataset_rows}
    retrieval_ids = set(retrieved_map.keys())

    missing = sorted(dataset_ids - retrieval_ids)
    extra = sorted(retrieval_ids - dataset_ids)

    if missing:
        preview = missing[:10]
        raise ValueError(f"Missing retrieved_evidence for {len(missing)} claim(s). Examples: {preview}")

    if extra:
        print(f"Warning: retrieval file has {len(extra)} extra claim_id(s) not used by dataset.")

    return True

In [10]:
dataset = load_jsonl(DATASET_PATHS[RUN_TAG])
if MAX_SAMPLES is not None:
    dataset = dataset[:MAX_SAMPLES]

retrieval_rows = load_jsonl(RETRIEVED_EVIDENCE_PATHS[RUN_TAG])

validate_dataset(dataset)
validate_retrieval_rows(retrieval_rows)
retrieved_map = build_retrieved_map(retrieval_rows)
validate_dataset_retrieval_match(dataset, retrieved_map)

print("RUN_TAG:", RUN_TAG)
print("Dataset size:", len(dataset))
print("Retrieved map size:", len(retrieved_map))
print("\nLabel distribution:")
print(pd.Series([row["label"] for row in dataset]).value_counts())
print("\nClaim type distribution:")
print(pd.Series([row["claim_type"] for row in dataset]).value_counts())

RUN_TAG: dev
Dataset size: 264
Retrieved map size: 264

Label distribution:
SUPPORTED              88
REFUTED                88
NOT_ENOUGH_EVIDENCE    88
Name: count, dtype: int64

Claim type distribution:
MILITARY_DIPLOMACY          101
PERSON_ROLE                  52
TIME_EVENT                   42
POLITICAL_ADMINISTRATIVE     35
GENEALOGY_SUCCESSION         22
NATURAL_OMEN_DISASTER        12
Name: count, dtype: int64


## 2. Format evidence và tạo prompt

Notebook dùng trực tiếp các field cố định:

- Gold evidence: `item["raw_text"]`, `item["pages"]`, `item["evidence_ids"]`.
- Retrieved evidence: `ev["rank"]`, `ev["chunk_id"]`, `ev["pages"]`, `ev["raw_text"]`.

Không dùng fallback nhiều key. Không truncate evidence. Không đưa `score`, `hybrid_score`, `bm25_score`, `dense_score` vào prompt verifier.

In [11]:
def format_gold_evidence(item):
    pages = item["pages"]
    raw_text = item["raw_text"].strip()
    return f"[Gold evidence: pages={pages}]\n{raw_text}"


def get_retrieval_info(item):
    claim_id = str(item["claim_id"])
    evidences = retrieved_map[claim_id][:MAX_EVIDENCE_ITEMS]

    gold_evidence_ids = {str(x) for x in item["evidence_ids"]}
    retrieved_chunk_ids = [str(ev["chunk_id"]) for ev in evidences]

    gold_rank = None
    for ev in evidences:
        ev_chunk_id = str(ev["chunk_id"])
        if ev_chunk_id in gold_evidence_ids:
            try:
                gold_rank = int(ev["rank"])
            except Exception:
                gold_rank = ev["rank"]
            break

    retrieval_hit = gold_rank is not None
    missing_retrieval = len(evidences) == 0

    return {
        "evidences": evidences,
        "gold_evidence_ids": sorted(gold_evidence_ids),
        "retrieved_chunk_ids": retrieved_chunk_ids,
        "retrieval_hit": retrieval_hit,
        "gold_rank": gold_rank,
        "missing_retrieval": missing_retrieval,
    }


def format_retrieved_evidence(item):
    info = get_retrieval_info(item)
    blocks = []

    for ev in info["evidences"]:
        rank = ev["rank"]
        pages = ev["pages"]
        raw_text = ev["raw_text"].strip()
        blocks.append(f"[Evidence {rank}: pages={pages}]\n{raw_text}")

    return "\n\n".join(blocks), info


def build_prompt(item, setting):
    claim = item["claim"].strip()

    if setting == "claim_only":
        return f"""Bạn là hệ thống kiểm chứng thông tin lịch sử tiếng Việt.

Nhiệm vụ: phân loại claim thành đúng một trong ba nhãn:
- SUPPORTED: claim có thể được xem là đúng.
- REFUTED: claim có lỗi sai trọng tâm.
- NOT_ENOUGH_EVIDENCE: không đủ chắc chắn để xác nhận hoặc bác bỏ.

Không có evidence được cung cấp trong setting này. Nếu không đủ chắc chắn, hãy chọn NOT_ENOUGH_EVIDENCE.

Claim:
{claim}

Chỉ trả về một JSON duy nhất. Không giải thích. Không dùng markdown.
{{"label":"SUPPORTED"}}
"""

    if setting == "gold_evidence":
        evidence_text = format_gold_evidence(item)
        return f"""Bạn là hệ thống kiểm chứng thông tin lịch sử tiếng Việt dựa trên evidence.

Chỉ dùng evidence được cung cấp. Không dùng kiến thức ngoài.

Nhãn:
- SUPPORTED: evidence đủ xác nhận claim.
- REFUTED: evidence mâu thuẫn trực tiếp với một chi tiết trọng tâm của claim.
- NOT_ENOUGH_EVIDENCE: evidence liên quan nhưng thiếu chi tiết quan trọng để xác nhận hoặc bác bỏ claim.

Claim:
{claim}

Evidence:
{evidence_text}

Chỉ trả về một JSON duy nhất. Không giải thích. Không dùng markdown.
{{"label":"SUPPORTED"}}
"""

    if setting == "retrieved_evidence":
        evidence_text, _ = format_retrieved_evidence(item)
        return f"""Bạn là hệ thống kiểm chứng thông tin lịch sử tiếng Việt dựa trên retrieved evidence.

Chỉ dùng retrieved evidence được cung cấp. Không dùng kiến thức ngoài. Retrieved evidence có thể thiếu, nhiễu, hoặc không chứa bằng chứng đúng.

Nhãn:
- SUPPORTED: retrieved evidence đủ xác nhận claim.
- REFUTED: retrieved evidence đủ bác bỏ một chi tiết trọng tâm của claim.
- NOT_ENOUGH_EVIDENCE: retrieved evidence không đủ để xác nhận hoặc bác bỏ claim.

Claim:
{claim}

Retrieved evidence:
{evidence_text}

Chỉ trả về một JSON duy nhất. Không giải thích. Không dùng markdown.
{{"label":"SUPPORTED"}}
"""

    raise ValueError(f"Unknown setting: {setting}")

## 3. Parse output và gọi LLM

In [12]:
def parse_label(raw_text):
    text = (raw_text or "").strip()
    candidates = []

    # Trường hợp model vẫn trả code block JSON.
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        candidates.append(fence.group(1).strip())

    # Trường hợp model trả một object JSON nằm trong text.
    json_obj = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if json_obj:
        candidates.append(json_obj.group(0))

    candidates.append(text)

    for cand in candidates:
        try:
            obj = json.loads(cand)
            label = str(obj.get("label", "")).strip().upper()
            if label in LABELS:
                return label, False
        except Exception:
            pass

    # Fallback cuối: nếu có xuất hiện nhãn rõ ràng trong text thì lấy nhãn đó.
    match = re.search(r"NOT_ENOUGH_EVIDENCE|SUPPORTED|REFUTED", text.upper())
    if match:
        return match.group(0), False

    # Không tự gán về NOT_ENOUGH_EVIDENCE để tránh làm lệch kết quả.
    return "PARSE_ERROR", True


def get_input_device(model):
    device_map = getattr(model, "hf_device_map", None)
    if isinstance(device_map, dict):
        for dev in device_map.values():
            if dev in ["cpu", "disk", "meta"]:
                continue
            if isinstance(dev, int):
                return torch.device(f"cuda:{dev}")
            if isinstance(dev, str) and dev.startswith("cuda"):
                return torch.device(dev)
    return next(model.parameters()).device


def render_chat(tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


def generate_one(tokenizer, model, prompt):
    text = render_chat(tokenizer, prompt)
    input_device = get_input_device(model)

    # Không tự truncate evidence ở bước format. Đây chỉ là giới hạn an toàn cho tokenizer.
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).to(input_device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 4. Load model

In [13]:
def get_quant_config(quantization):
    if quantization == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    if quantization == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    return None


def load_llm(model_name, quantization):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=get_quant_config(quantization),
        device_map="auto",
        trust_remote_code=True,
        torch_dtype="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tokenizer, model


def cleanup_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

## 5. Chạy dự đoán theo từng model và setting

In [14]:
def run_one_setting(tokenizer, model, model_tag, model_name, quantization, setting):
    rows = []

    for item in tqdm(dataset, desc=f"{model_tag} | {setting}"):
        claim_id = str(item["claim_id"])
        prompt = build_prompt(item, setting)
        raw_response = generate_one(tokenizer, model, prompt)
        predicted_label, parse_error = parse_label(raw_response)

        retrieval_info = {
            "gold_evidence_ids": [str(x) for x in item["evidence_ids"]],
            "retrieved_chunk_ids": [],
            "retrieval_hit": None,
            "gold_rank": None,
            "missing_retrieval": False,
        }

        if setting == "retrieved_evidence":
            retrieval_info = get_retrieval_info(item)

        rows.append({
            "claim_id": claim_id,
            "claim": item["claim"],
            "gold_label": item["label"],
            "predicted_label": predicted_label,
            "model_tag": model_tag,
            "model_name": model_name,
            "quantization": quantization,
            "setting": setting,
            "raw_response": raw_response,
            "parse_error": bool(parse_error),
            "claim_type": item["claim_type"],
            "split": item["split"],
            "gold_chunk_id": str(item["chunk_id"]),
            "gold_evidence_ids": retrieval_info["gold_evidence_ids"],
            "retrieved_chunk_ids": retrieval_info["retrieved_chunk_ids"],
            "retrieval_hit": retrieval_info["retrieval_hit"],
            "gold_rank": retrieval_info["gold_rank"],
            "missing_retrieval": bool(retrieval_info["missing_retrieval"]),
        })

    return rows

In [15]:
all_predictions = []

for cfg in MODELS:
    print("\n" + "=" * 80)
    print("Model:", cfg)

    pending_settings = []
    for setting in SETTINGS:
        out_path = OUTPUT_DIR / f"{RUN_TAG}_{cfg['tag']}_{setting}_predictions.jsonl"

        if out_path.exists() and not FORCE_RERUN:
            print("Skip existing:", out_path)
            rows = load_jsonl(out_path)
            all_predictions.extend(rows)
        else:
            pending_settings.append(setting)

    if not pending_settings:
        continue

    tokenizer = None
    model = None

    try:
        tokenizer, model = load_llm(cfg["name"], cfg["quantization"])

        for setting in pending_settings:
            out_path = OUTPUT_DIR / f"{RUN_TAG}_{cfg['tag']}_{setting}_predictions.jsonl"
            rows = run_one_setting(
                tokenizer=tokenizer,
                model=model,
                model_tag=cfg["tag"],
                model_name=cfg["name"],
                quantization=cfg["quantization"],
                setting=setting,
            )
            save_jsonl(rows, out_path)
            print("Saved:", out_path)
            all_predictions.extend(rows)

    finally:
        print("Cleaning up model:", cfg["tag"])
        cleanup_model(model, tokenizer)

all_path = OUTPUT_DIR / f"{RUN_TAG}_verification_predictions_all.jsonl"
save_jsonl(all_predictions, all_path)
print("Saved all:", all_path)
print("Total predictions:", len(all_predictions))


Model: {'tag': 'qwen25_3b_8bit', 'name': 'Qwen/Qwen2.5-3B-Instruct', 'quantization': '8bit'}


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Saved: /kaggle/working/datasets/lanthanhvi/verification_outputs_clean/dev_qwen25_7b_8bit_claim_only_predictions.jsonl


qwen25_7b_8bit | gold_evidence:   0%|          | 0/264 [00:00<?, ?it/s]

Saved: /kaggle/working/datasets/lanthanhvi/verification_outputs_clean/dev_qwen25_7b_8bit_gold_evidence_predictions.jsonl


qwen25_7b_8bit | retrieved_evidence:   0%|          | 0/264 [00:00<?, ?it/s]

Saved: /kaggle/working/datasets/lanthanhvi/verification_outputs_clean/dev_qwen25_7b_8bit_retrieved_evidence_predictions.jsonl
Cleaning up model: qwen25_7b_8bit
Saved all: /kaggle/working/datasets/lanthanhvi/verification_outputs_clean/dev_verification_predictions_all.jsonl
Total predictions: 1584


## 6. Metric tổng hợp

In [16]:
def compute_metric_row(group):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()

    per_label = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)

    return {
        "model_tag": group["model_tag"].iloc[0],
        "model_name": group["model_name"].iloc[0],
        "quantization": group["quantization"].iloc[0],
        "setting": group["setting"].iloc[0],
        "num_samples": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "f1_supported": per_label[0],
        "f1_refuted": per_label[1],
        "f1_not_enough_evidence": per_label[2],
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
        "missing_retrieval_count": int(group["missing_retrieval"].fillna(False).sum()),
    }


df = pd.DataFrame(all_predictions)
if df.empty:
    raise ValueError("No predictions found. Run prediction cells first.")

summary_rows = []
for _, group in df.groupby(["model_tag", "setting"], sort=False):
    summary_rows.append(compute_metric_row(group))

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / f"{RUN_TAG}_verification_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df

,model_tag,model_name,quantization,setting,num_samples,accuracy,macro_f1,f1_supported,f1_refuted,f1_not_enough_evidence,parse_error_count,missing_retrieval_count
0,qwen25_3b_8bit,Qwen/Qwen2.5-3B-Instruct,8bit,claim_only,264,0.329545,0.165242,0.000000,0.000000,0.495726,0,0
1,qwen25_3b_8bit,Qwen/Qwen2.5-3B-Instruct,8bit,gold_evidence,264,0.469697,0.406828,0.576512,0.178218,0.465753,0,0
2,qwen25_3b_8bit,Qwen/Qwen2.5-3B-Instruct,8bit,retrieved_evidence,264,0.488636,0.464310,0.506329,0.338983,0.547619,0,0
3,qwen25_7b_8bit,Qwen/Qwen2.5-7B-Instruct,8bit,claim_only,264,0.333333,0.186392,0.043011,0.021978,0.494186,0,0
4,qwen25_7b_8bit,Qwen/Qwen2.5-7B-Instruct,8bit,gold_evidence,264,0.636364,0.575843,0.751092,0.285714,0.690722,0,0
5,qwen25_7b_8bit,Qwen/Qwen2.5-7B-Instruct,8bit,retrieved_evidence,264,0.609848,0.548901,0.748815,0.250000,0.647887,0,0


## 7. Confusion matrix

Confusion matrix dùng thêm cột `PARSE_ERROR` để không làm mất các output không parse được.

In [17]:
cm_rows = []

for (model_tag, setting), group in df.groupby(["model_tag", "setting"], sort=False):
    cm = confusion_matrix(group["gold_label"], group["predicted_label"], labels=EXTENDED_LABELS)
    for i, true_label in enumerate(EXTENDED_LABELS):
        for j, pred_label in enumerate(EXTENDED_LABELS):
            cm_rows.append({
                "model_tag": model_tag,
                "setting": setting,
                "true_label": true_label,
                "predicted_label": pred_label,
                "count": int(cm[i, j]),
            })

cm_df = pd.DataFrame(cm_rows)
cm_path = OUTPUT_DIR / f"{RUN_TAG}_verification_confusion_matrix.csv"
cm_df.to_csv(cm_path, index=False)
cm_df.head(30)

,model_tag,setting,true_label,predicted_label,count
0,qwen25_3b_8bit,claim_only,SUPPORTED,SUPPORTED,0
1,qwen25_3b_8bit,claim_only,SUPPORTED,REFUTED,0
2,qwen25_3b_8bit,claim_only,SUPPORTED,NOT_ENOUGH_EVIDENCE,88
3,qwen25_3b_8bit,claim_only,SUPPORTED,PARSE_ERROR,0
4,qwen25_3b_8bit,claim_only,REFUTED,SUPPORTED,0
5,qwen25_3b_8bit,claim_only,REFUTED,REFUTED,0
6,qwen25_3b_8bit,claim_only,REFUTED,NOT_ENOUGH_EVIDENCE,88
7,qwen25_3b_8bit,claim_only,REFUTED,PARSE_ERROR,0
8,qwen25_3b_8bit,claim_only,NOT_ENOUGH_EVIDENCE,SUPPORTED,1
9,qwen25_3b_8bit,claim_only,NOT_ENOUGH_EVIDENCE,REFUTED,0


## 8. Metric theo claim_type

In [18]:
claim_type_rows = []

for (model_tag, setting, claim_type), group in df.groupby(["model_tag", "setting", "claim_type"], sort=False):
    y_true = group["gold_label"].astype(str).tolist()
    y_pred = group["predicted_label"].astype(str).tolist()

    claim_type_rows.append({
        "model_tag": model_tag,
        "setting": setting,
        "claim_type": claim_type,
        "num_samples": len(group),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
        "parse_error_count": int(group["parse_error"].fillna(False).sum()),
    })

claim_type_df = pd.DataFrame(claim_type_rows)
claim_type_path = OUTPUT_DIR / f"{RUN_TAG}_verification_claim_type_metrics.csv"
claim_type_df.to_csv(claim_type_path, index=False)
claim_type_df.head(30)

,model_tag,setting,claim_type,num_samples,accuracy,macro_f1,parse_error_count
0,qwen25_3b_8bit,claim_only,PERSON_ROLE,52,0.269231,0.141414,0
1,qwen25_3b_8bit,claim_only,TIME_EVENT,42,0.595238,0.248756,0
2,qwen25_3b_8bit,claim_only,MILITARY_DIPLOMACY,101,0.247525,0.132275,0
3,qwen25_3b_8bit,claim_only,NATURAL_OMEN_DISASTER,12,0.250000,0.133333,0
4,qwen25_3b_8bit,claim_only,GENEALOGY_SUCCESSION,22,0.090909,0.055556,0
5,qwen25_3b_8bit,claim_only,POLITICAL_ADMINISTRATIVE,35,0.514286,0.226415,0
6,qwen25_3b_8bit,gold_evidence,PERSON_ROLE,52,0.403846,0.308197,0
7,qwen25_3b_8bit,gold_evidence,TIME_EVENT,42,0.476190,0.494052,0
8,qwen25_3b_8bit,gold_evidence,MILITARY_DIPLOMACY,101,0.475248,0.414157,0
9,qwen25_3b_8bit,gold_evidence,NATURAL_OMEN_DISASTER,12,0.416667,0.395238,0


## 9. Phân tích retrieved_evidence theo retrieval_hit

Phần này chỉ áp dụng cho setting `retrieved_evidence`.

- `retrieval_hit = True`: top-5 retrieved evidence có chứa ít nhất một `gold_evidence_ids`.
- `gold_rank`: vị trí đầu tiên của gold evidence trong top-5.

Nếu `retrieval_hit=True` nhưng verifier sai, lỗi nghiêng về verifier/prompt/model. Nếu `retrieval_hit=False` và verifier sai, lỗi có thể bắt nguồn từ retrieval.

In [19]:
retrieved_df = df[df["setting"] == "retrieved_evidence"].copy()

retrieval_hit_rows = []
if not retrieved_df.empty:
    for (model_tag, retrieval_hit), group in retrieved_df.groupby(["model_tag", "retrieval_hit"], dropna=False, sort=False):
        y_true = group["gold_label"].astype(str).tolist()
        y_pred = group["predicted_label"].astype(str).tolist()
        retrieval_hit_rows.append({
            "model_tag": model_tag,
            "retrieval_hit": retrieval_hit,
            "num_samples": len(group),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0),
            "parse_error_count": int(group["parse_error"].fillna(False).sum()),
        })

retrieval_hit_df = pd.DataFrame(retrieval_hit_rows)
retrieval_hit_path = OUTPUT_DIR / f"{RUN_TAG}_verification_by_retrieval_hit.csv"
retrieval_hit_df.to_csv(retrieval_hit_path, index=False)
retrieval_hit_df

,model_tag,retrieval_hit,num_samples,accuracy,macro_f1,parse_error_count
0,qwen25_3b_8bit,True,244,0.500000,0.478756,0
1,qwen25_3b_8bit,False,20,0.350000,0.250000,0
2,qwen25_7b_8bit,True,244,0.627049,0.565038,0
3,qwen25_7b_8bit,False,20,0.400000,0.281905,0


In [20]:
# File chẩn đoán theo từng claim cho setting retrieved_evidence.
retrieval_diag_cols = [
    "model_tag", "claim_id", "gold_label", "predicted_label", "claim_type",
    "gold_evidence_ids", "retrieved_chunk_ids", "retrieval_hit", "gold_rank", "parse_error"
]

retrieval_diag_df = retrieved_df[retrieval_diag_cols].copy() if not retrieved_df.empty else pd.DataFrame(columns=retrieval_diag_cols)
retrieval_diag_path = OUTPUT_DIR / f"{RUN_TAG}_verification_retrieval_diagnostics.csv"
retrieval_diag_df.to_csv(retrieval_diag_path, index=False)
retrieval_diag_df.head(20)

,model_tag,claim_id,gold_label,predicted_label,claim_type,gold_evidence_ids,retrieved_chunk_ids,retrieval_hit,gold_rank,parse_error
528,qwen25_3b_8bit,claim_000013,SUPPORTED,SUPPORTED,PERSON_ROLE,[019f17e1-0a34-793b-b170-d28a55c14360],"[019f17e1-0a34-793b-b170-d28a55c14360, 019f17e...",True,1.0,False
529,qwen25_3b_8bit,claim_000014,REFUTED,NOT_ENOUGH_EVIDENCE,PERSON_ROLE,[019f17e1-0a34-793b-b170-d28a55c14360],"[019f17e1-0a34-793b-b170-d28a55c14360, 019f17e...",True,1.0,False
530,qwen25_3b_8bit,claim_000015,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,TIME_EVENT,[019f17e1-0a34-793b-b170-d28a55c14360],"[019f17e1-0a34-793b-b170-d28a55c14360, 019f17e...",True,1.0,False
531,qwen25_3b_8bit,claim_000019,SUPPORTED,NOT_ENOUGH_EVIDENCE,MILITARY_DIPLOMACY,[019f17e1-16f9-7209-b68a-a78071add7d2],"[019f17e1-16f9-7209-b68a-a78071add7d2, 019f17e...",True,1.0,False
532,qwen25_3b_8bit,claim_000020,REFUTED,NOT_ENOUGH_EVIDENCE,NATURAL_OMEN_DISASTER,[019f17e1-16f9-7209-b68a-a78071add7d2],"[019f17e1-16f9-7209-b68a-a78071add7d2, 019f17e...",True,1.0,False
533,qwen25_3b_8bit,claim_000021,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,MILITARY_DIPLOMACY,[019f17e1-16f9-7209-b68a-a78071add7d2],"[019f17e1-16f9-7209-b68a-a78071add7d2, 019f17e...",True,1.0,False
534,qwen25_3b_8bit,claim_000037,SUPPORTED,NOT_ENOUGH_EVIDENCE,NATURAL_OMEN_DISASTER,[019f17e1-3b52-74b1-9868-5087c9550a67],"[019f17e1-3b52-74b1-9868-5087c9550a67, 019f17e...",True,1.0,False
535,qwen25_3b_8bit,claim_000038,REFUTED,REFUTED,MILITARY_DIPLOMACY,[019f17e1-3b52-74b1-9868-5087c9550a67],"[019f17e1-3b52-74b1-9868-5087c9550a67, 019f17e...",True,1.0,False
536,qwen25_3b_8bit,claim_000039,NOT_ENOUGH_EVIDENCE,NOT_ENOUGH_EVIDENCE,MILITARY_DIPLOMACY,[019f17e1-3b52-74b1-9868-5087c9550a67],"[019f17e5-f899-7920-aecc-3656ffdd77f7, 019f17e...",True,2.0,False
537,qwen25_3b_8bit,claim_000067,SUPPORTED,NOT_ENOUGH_EVIDENCE,PERSON_ROLE,[019f17e1-7757-7050-be40-3233e71193d3],"[019f17e1-7757-7050-be40-3233e71193d3, 019f17e...",True,1.0,False


## 10. Nén output

In [21]:
zip_base = Path(f"/kaggle/working/verification_outputs_clean_{RUN_TAG}")
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)

print("ZIP:", zip_path)
print("Files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)

ZIP: /kaggle/working/verification_outputs_clean_dev.zip
Files:
- dev_qwen25_3b_8bit_claim_only_predictions.jsonl
- dev_qwen25_3b_8bit_gold_evidence_predictions.jsonl
- dev_qwen25_3b_8bit_retrieved_evidence_predictions.jsonl
- dev_qwen25_7b_8bit_claim_only_predictions.jsonl
- dev_qwen25_7b_8bit_gold_evidence_predictions.jsonl
- dev_qwen25_7b_8bit_retrieved_evidence_predictions.jsonl
- dev_verification_by_retrieval_hit.csv
- dev_verification_claim_type_metrics.csv
- dev_verification_confusion_matrix.csv
- dev_verification_predictions_all.jsonl
- dev_verification_retrieval_diagnostics.csv
- dev_verification_summary.csv
